# Best Practices and Pitfalls

*Last updated: June 2026*

This page collects known limitations, non-obvious configuration requirements, and
recommendations gathered from experience with JAX-ALFA simulations. Read through
these before setting up a new case — each item documents a real pitfall that is
easy to overlook.

---
## 1. Surface Flux as Lower Boundary Condition for Stable Boundary Layers

The surface sensible heat flux (SHFX) exhibits a **dual nature** in stably stratified conditions: a single value of SHFX can correspond to two distinct stability states: weakly stable and very stable regimes. This fundamental property was first identified by Malhi (1995) and later elaborated by Basu et al. (2008).

When SHFX is prescribed as the lower boundary condition, a PBL model (single-column or LES) can **only access the weakly stable solution**. The very stable regime — where turbulence is strongly suppressed — remains inaccessible under a flux prescription. To simulate both weakly stable and very stable conditions faithfully, the **surface temperature** must be prescribed instead.

Set ``optSurfBC = 2`` in ``Config.py`` to prescribe a time-varying surface temperature. This is also the approach adopted in all GABLS intercomparison studies.

**References:**

Malhi, Y. S. (1995). The significance of the dual solutions for heat fluxes measured by the temperature fluctuations method in stable conditions. *Boundary-Layer Meteorology*, 74, 389–396.

Basu, S., Holtslag, A. A. M., Van De Wiel, B. J. H., Moene, A. F., and Steeneveld, G.-J. (2008). An inconvenient "truth" about using sensible heat flux as a surface boundary condition in models under stably stratified regimes. *Acta Geophysica*, 56(1), 88–99. https://doi.org/10.2478/s11600-007-0038-y

---
## 2. First Half-Model Level Height for High-Resolution Simulations

### Physical basis

MOST-based boundary conditions are valid only within the **inertial sublayer** (ISL), whose lower limit coincides with the top of the roughness sublayer (RSL) at height $z_* = 50 z_0$. A number of recent high-resolution LES studies have placed $z_1$ **below** this limit and incorrectly invoked MOST, resulting in erroneous surface fluxes and artificial resolution sensitivity.

### Practical implication

The first half-model level sits at $z_1 = \Delta z / 2$. Requiring $z_1 > 50\,z_0$ imposes a lower bound on the grid spacing:

$$
\Delta z > 100\,z_0.
$$

For the GABLS1 case ($z_0 = 0.1$ m) this means $\Delta z > 10$ m and $z_1 > 5$ m, which limits how far the grid can be refined before MOST breaks down.

### Temporary workaround

Reduce $z_0$ in ``Config.py`` to restore the $z_1 > 50\,z_0$ constraint without changing the grid resolution. This is a **numerical fix, not a physical approach**.

### Permanent solution

Adopt a **roughness sublayer (RSL) parameterization** that replaces MOST at heights $z_1 < z_*$. Several formulations exist (e.g., Physick and Garratt 1995; Arnqvist and Bergström 2015), and their incorporation into LES codes is straightforward. RSL models are currently under development for JAX-ALFA.

**Reference:**
Basu, S., and Lacser, A. (2017). A cautionary note on the use of Monin–Obukhov similarity theory in very high-resolution large-eddy simulations. *Boundary-Layer Meteorology*, 163, 197–207.
https://doi.org/10.1007/s10546-016-0225-y

---
## 3. Filter-to-Grid Ratio (FGR) for LASDD-SM in Stable Boundary Layers

Set ``FGR = 2`` in ``Config.py``. This is the value used in the original LASDD-SM
paper (Basu and Porté-Agel 2006), and it is required for stable BL simulations.

```python
FGR = 2
```

As an added benefit, when ``FGR = 2`` the explicit filter acts as a dealiasing
mechanism, so the 3/2-rule zero-padding step is skipped and the simulation runs
slightly faster.

**Reference:**

Basu, S., and Porté-Agel, F. (2006). Large-eddy simulation of stably stratified
atmospheric boundary layer turbulence: A scale-dependent dynamic modeling approach.
*Journal of the Atmospheric Sciences*, 63(8), 2074–2091.
https://doi.org/10.1175/JAS3734.1